In [ ]:
import numpy as np
import scipy.signal as signal
import scanpy as sc
from tqdm import tqdm
import pandas as pd
import anndata as ad

from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d
from tqdm import tqdm


In [ ]:
def create_global_spectrum(X, mz_axis):
    # Sum all spectra (X is shape: [n_pixels, n_mz])
    global_spectrum = np.sum(X, axis=0)
    return mz_axis, global_spectrum

def detect_peaks(mz, intensity, min_prominence=0.01, smoothing_sigma=1):
    # Smooth the spectrum
    smoothed = gaussian_filter1d(intensity, sigma=smoothing_sigma)
    peaks, _ = find_peaks(smoothed, prominence=min_prominence * np.max(smoothed))
    return mz[peaks]

def detect_peaks(spectrum, mz_axis, prominence=0.01, intensity_threshold=0.0):
    """Detect peaks with prominence and apply intensity threshold."""
    peaks, properties = signal.find_peaks(spectrum, prominence=prominence)
    intensities = spectrum[peaks]

    # Filter peaks based on intensity threshold
    valid = intensities >= intensity_threshold
    return mz_axis[peaks][valid], intensities[valid]

def mz_switch_calculator(ppm_tolerance, da_tolerance):
    return da_tolerance/ppm_tolerance *1e6


def ppm_diff(ref_mz, target_mz):
    return (target_mz - ref_mz) / ref_mz * 1e6


def find_top_peaks_global(adata, prominence=0.01, n_peaks=100, ppm_tolerance=10, da_tolerance=0.01,
                          mz_switch=500.0, intensity_threshold=0.0):
    all_peaks = []
    mz_axis = adata.var_names.astype(float).values

    for i in tqdm(range(adata.shape[0]), desc="Detecting peaks"):
        spectrum = adata.X[i].toarray().flatten()
        mz_peaks, intensities = detect_peaks(spectrum, mz_axis, prominence, intensity_threshold)
        all_peaks.extend(list(zip(mz_peaks, intensities)))

    # Convert to structured array and sort by intensity
    all_peaks = np.array(all_peaks, dtype=[("mz", float), ("intensity", float)])
    sorted_peaks = np.sort(all_peaks, order="intensity")[::-1]

    # Select top N, removing duplicates within ±ppm_tolerance
    selected_peaks = []
    for peak in sorted_peaks:
        if len(selected_peaks) >= n_peaks:
            break

        def is_far_enough(p):
            if peak["mz"] < mz_switch:
                return abs(p["mz"] - peak["mz"]) > da_tolerance
            else:
                return abs(ppm_diff(p["mz"], peak["mz"])) > ppm_tolerance

        if all(is_far_enough(p) for p in selected_peaks):
            selected_peaks.append(peak)

    return np.sort(np.array([p["mz"] for p in selected_peaks]))

def extract_and_aggregate_peaks(adata, selected_mz, ppm_tolerance=10, da_tolerance=0.01, mz_switch=500.0):
    mz_axis = adata.var_names.astype(float).values
    X = adata.X

    aggregated_intensities = []
    aggregated_mz_names = []

    for target_mz in selected_mz:
        # Find all indices within the ppm tolerance
        if target_mz < mz_switch:
            diff_array = np.abs(mz_axis - target_mz)
            within_tolerance = np.where(diff_array <= da_tolerance)[0]
        else:
            ppm_diff_array = np.abs((mz_axis - target_mz) / target_mz * 1e6)
            within_tolerance = np.where(ppm_diff_array <= ppm_tolerance)[0]

        if len(within_tolerance) == 0:
            # No peaks found within tolerance, fill with zeros
            summed_intensity = np.zeros((adata.shape[0],))
            avg_mz = target_mz
        else:
            # Sum intensities across matched peaks
            summed = X[:, within_tolerance].sum(axis=1)
            summed_intensity = np.asarray(summed).flatten()
            avg_mz = mz_axis[within_tolerance].mean()

        aggregated_intensities.append(summed_intensity)
        aggregated_mz_names.append(f"{avg_mz:.6f}")
    
    # Stack the results to form a new data matrix
    new_X = np.vstack(aggregated_intensities).T # shape: (n_obs, n_selected_mz)

    # Create new AnnData object
    new_adata = ad.AnnData(
        X=new_X,
        obs=adata.obs.copy(),
        var=pd.DataFrame(index=aggregated_mz_names)
    )

    return new_adata

In [4]:
# === File Paths ===
input_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a.h5ad"
output_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_peaks_010_1000_165_0015_1000.h5ad"

# === Parameters ===
prominence = 0.01
top_n = 1000
ppm_tolerance = 16.5
da_tolerance = 0.015
mz_switch = 500.0
intensity_threshold = 1000.0
mz_switch = mz_switch_calculator(ppm_tolerance, da_tolerance)

# === Load and process ===
adata = sc.read_h5ad(input_file)
print(f"Loaded AnnData from: {input_file}")

selected_mz = find_top_peaks_global(
    adata,
    prominence=prominence,
    n_peaks=top_n,
    ppm_tolerance=ppm_tolerance,
    da_tolerance=da_tolerance,
    mz_switch=mz_switch,
    intensity_threshold=intensity_threshold
)

adata_top_peaks = extract_and_aggregate_peaks(
    adata,
    selected_mz,
    ppm_tolerance=ppm_tolerance,
    da_tolerance=da_tolerance,
    mz_switch=mz_switch
)

adata_top_peaks.write(output_file)
print(f"Reduced AnnData with top {top_n} peaks saved to: {output_file}")

/opt/anaconda3/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Loaded AnnData from: /Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a.h5ad


Detecting peaks: 100%|██████████| 16605/16605 [01:15<00:00, 219.74it/s]


Reduced AnnData with top 1000 peaks saved to: /Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_peaks_010_1000_165_0015_1000.h5ad


In [5]:
input_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_peaks_010_1000_165_0015_1000.h5ad"
adata = sc.read_h5ad(input_file)

In [6]:
print(adata)

AnnData object with n_obs × n_vars = 16605 × 852
    obs: 'x', 'y', 'TIC', 'sample', 'batch', 'age_group', 'disease_status'


In [7]:
adata.var

""
96.926096
112.898041
114.892378
136.060809
137.025089
...
1947.522664
1949.681396
1950.201904
1955.064087


In [8]:
print(adata)

AnnData object with n_obs × n_vars = 16605 × 852
    obs: 'x', 'y', 'TIC', 'sample', 'batch', 'age_group', 'disease_status'


In [9]:
adata

AnnData object with n_obs × n_vars = 16605 × 852
    obs: 'x', 'y', 'TIC', 'sample', 'batch', 'age_group', 'disease_status'

In [10]:
import plotly.graph_objects as go
import numpy as np

def plot_top_peaks_stem(adata, title="Top 1000 Peaks by Max Intensity (Stem Plot)"):
    # Get m/z values
    mz_axis = adata.var_names.astype(float).values

    # Get max intensity per m/z (not sum)
    if hasattr(adata.X, "toarray"):  # sparse
        max_intensity = np.asarray(adata.X.max(axis=0)).ravel()
    else:  # dense
        max_intensity = np.max(adata.X, axis=0)

    # Find top 1000 peaks by max intensity
    top_indices = np.argsort(max_intensity)[::-1][:1000]
    mz_top = mz_axis[top_indices]
    intensity_top = max_intensity[top_indices]

    # Sort m/z values for cleaner appearance
    sorted_order = np.argsort(mz_top)
    mz_top = mz_top[sorted_order]
    intensity_top = intensity_top[sorted_order]

    # Create stem plot (vertical lines + markers)
    fig = go.Figure()

    # Vertical lines (stems)
    for x, y in zip(mz_top, intensity_top):
        fig.add_trace(go.Scatter(
            x=[x, x],
            y=[0, y],
            mode='lines',
            line=dict(color='crimson', width=1),
            showlegend=False,
            hoverinfo='skip'
        ))

    # Markers (dots at the top of each stem)
    fig.add_trace(go.Scatter(
        x=mz_top,
        y=intensity_top,
        mode='markers',
        marker=dict(size=4, color='crimson'),
        hovertemplate='Avg m/z: %{x:.4f}<br>Max Intensity: %{y:.2f}',
        name='Max Intensity'
    ))

    fig.update_layout(
        title=title,
        xaxis_title="Averaged m/z",
        yaxis_title="Max Intensity (across sample)",
        template="plotly_white",
        dragmode="zoom",
        hovermode="closest"
    )

    fig.show()


In [11]:
plot_top_peaks_stem(adata)

In [12]:
import plotly.graph_objects as go
import numpy as np

def plot_top_peaks_by_max_intensity(adata, title="Top 1000 Peaks by Max Intensity"):
    # Get m/z values
    mz_axis = adata.var_names.astype(float).values

    # Get max intensity per m/z (not sum)
    if hasattr(adata.X, "toarray"):  # sparse
        max_intensity = np.asarray(adata.X.max(axis=0)).ravel()
    else:  # dense
        max_intensity = np.max(adata.X, axis=0)

    # Find top 1000 peaks by max intensity
    top_indices = np.argsort(max_intensity)[::-1][:1000]
    mz_top = mz_axis[top_indices]
    intensity_top = max_intensity[top_indices]

    # Sort m/z values for nicer plot appearance
    sorted_order = np.argsort(mz_top)
    mz_top = mz_top[sorted_order]
    intensity_top = intensity_top[sorted_order]

    # Plot using Plotly
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=mz_top,
        y=intensity_top,
        mode='lines+markers',
        marker=dict(size=4, color='crimson'),
        line=dict(width=1),
        hovertemplate='Avg m/z: %{x:.4f}<br>Max Intensity: %{y:.2f}',
        name='Max Intensity'
    ))

    fig.update_layout(
        title=title,
        xaxis_title="Averaged m/z",
        yaxis_title="Max Intensity (across sample)",
        template="plotly_white",
        dragmode="zoom",
        hovermode="closest"
    )

    fig.show()

In [13]:
plot_top_peaks_by_max_intensity(adata)


In [2]:
import anndata as ad
import numpy as np
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d
import os

# === Parameters ===
input_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a.h5ad"
output_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_peaks_020_010_10.h5ad"

window_Da = 0.02
min_prominence = 0.01  # 1% of global max
smoothing_sigma = 1
layer = None  # or "raw" if you want a specific layer

# === Load Data ===
adata = ad.read_h5ad(input_file)

# === Check for mz axis ===
if "mz" not in adata.var.columns:
    raise ValueError("AnnData object must have 'mz' in adata.var.")

mz_axis = adata.var["mzs"].values.astype(np.float64)
X = adata.X if layer is None else adata.layers[layer]

# === Create global spectrum ===
global_spectrum = np.sum(X, axis=0)

# === Detect peaks in global spectrum ===
smoothed = gaussian_filter1d(global_spectrum, sigma=smoothing_sigma)
peaks, _ = find_peaks(smoothed, prominence=min_prominence * np.max(smoothed))
global_peak_mz = mz_axis[peaks]

# === Integrate around peaks ===
def integrate_intensity_around_peaks(X, mz_axis, peak_list, window_Da):
    n_pixels = X.shape[0]
    n_peaks = len(peak_list)
    result = np.zeros((n_pixels, n_peaks), dtype=np.float32)

    for i, mz_target in enumerate(peak_list):
        mz_min = mz_target - window_Da / 2
        mz_max = mz_target + window_Da / 2
        mz_mask = (mz_axis >= mz_min) & (mz_axis <= mz_max)
        if np.any(mz_mask):
            result[:, i] = X[:, mz_mask].sum(axis=1)

    return result

binned_matrix = integrate_intensity_around_peaks(X, mz_axis, global_peak_mz, window_Da)

# === Save as a new layer and metadata ===
layer_name = f"hdi_{window_Da:.5f}Da"
adata.layers[layer_name] = binned_matrix
adata.uns[f"{layer_name}_mz"] = global_peak_mz

# === Save updated AnnData ===
adata.write_h5ad(output_file)

print(f"✅ Done. Global peaks: {len(global_peak_mz)} | Output saved to:\n{output_file}")

/opt/anaconda3/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


ValueError: AnnData object must have 'mz' in adata.var.

In [3]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import signal
from scipy.ndimage import gaussian_filter1d
import anndata as ad

def create_global_spectrum(X, mz_axis):
    global_spectrum = np.asarray(X.sum(axis=0)).flatten()
    return mz_axis, global_spectrum

def detect_global_peaks(mz_axis, global_spectrum, min_prominence=0.01, smoothing_sigma=1):
    #smoothed = gaussian_filter1d(global_spectrum, sigma=smoothing_sigma)
    #peaks, _ = signal.find_peaks(smoothed, prominence=min_prominence * np.max(smoothed))
    peaks, _ = signal.find_peaks(global_spectrum, prominence=min_prominence)

    return mz_axis[peaks]

def extract_intensity_within_tolerance(adata, global_peaks, da_tolerance=0.015):
    mz_axis = adata.var_names.astype(float).values
    X = adata.X

    extracted = []
    new_var_names = []

    for mz_target in global_peaks:
        mask = (mz_axis >= mz_target - da_tolerance / 2) & (mz_axis <= mz_target + da_tolerance / 2)
        if np.any(mask):
            summed = X[:, mask].sum(axis=1)
            extracted.append(np.asarray(summed).flatten())
            new_var_names.append(f"{mz_target:.6f}")

    new_X = np.vstack(extracted).T
    return ad.AnnData(X=new_X, obs=adata.obs.copy(), var=pd.DataFrame(index=new_var_names))


In [4]:
# === File Paths ===
input_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a.h5ad"
output_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_hdi_style.h5ad"

# === Parameters ===
prominence = 0.01
da_tolerance = 0.04
smoothing_sigma = 1

# === Load and Process ===
adata = sc.read_h5ad(input_file)
print(f"Loaded AnnData from: {input_file}")

mz_axis, global_spec = create_global_spectrum(adata.X, adata.var_names.astype(float).values)
global_peaks = detect_global_peaks(mz_axis, global_spec, min_prominence=prominence, smoothing_sigma=smoothing_sigma)
adata_hdi = extract_intensity_within_tolerance(adata, global_peaks, da_tolerance=da_tolerance)

adata_hdi.write(output_file)
print(f"Saved HDI-style AnnData to: {output_file}")

/opt/anaconda3/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Loaded AnnData from: /Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a.h5ad
Saved HDI-style AnnData to: /Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_hdi_style.h5ad


In [5]:
import plotly.graph_objects as go
import numpy as np

def plot_top_peaks_stem(adata, title="Top 1000 Peaks by Max Intensity (Stem Plot)"):
    mz_axis = adata.var_names.astype(float).values

    if hasattr(adata.X, "toarray"):
        max_intensity = np.asarray(adata.X.max(axis=0)).ravel()
    else:
        max_intensity = np.max(adata.X, axis=0)

    top_indices = np.argsort(max_intensity)[::-1][:1000]
    mz_top = mz_axis[top_indices]
    intensity_top = max_intensity[top_indices]

    sorted_order = np.argsort(mz_top)
    mz_top = mz_top[sorted_order]
    intensity_top = intensity_top[sorted_order]

    fig = go.Figure()

    for x, y in zip(mz_top, intensity_top):
        fig.add_trace(go.Scatter(
            x=[x, x],
            y=[0, y],
            mode='lines',
            line=dict(color='indigo', width=1),
            showlegend=False,
            hoverinfo='skip'
        ))

    fig.add_trace(go.Scatter(
        x=mz_top,
        y=intensity_top,
        mode='markers',
        marker=dict(size=4, color='indigo'),
        hovertemplate='m/z: %{x:.4f}<br>Max Intensity: %{y:.2f}',
        name='Top Peaks'
    ))

    fig.update_layout(
        title=title,
        xaxis_title="m/z",
        yaxis_title="Max Intensity (across all pixels)",
        template="plotly_white",
        hovermode="closest",
        dragmode="pan"
    )

    fig.show()

In [6]:
plot_top_peaks_stem(adata_hdi, title="Top 1000 Peaks by Max Intensity (Stem Plot)")


In [11]:
plot_top_peaks_stem(adata_hdi, title="Top 1000 Peaks by Max Intensity (Stem Plot)")


In [8]:
plot_top_peaks_stem(adata_hdi, title="Top 1000 Peaks by Max Intensity (Stem Plot)")
